In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os, shutil, json

ROOT_DIR        = "/content/drive/MyDrive/nutrition_rag"
PROCESSED_DIR   = os.path.join(ROOT_DIR, "data_processed")
VECTOR_DB_DIR   = os.path.join(ROOT_DIR, "vector_db_v4")
OUTPUT_DIR      = os.path.join(ROOT_DIR, "outputs")
ALL_RECORDS_FILE = os.path.join(PROCESSED_DIR, "all_records.json")

for d in [PROCESSED_DIR, VECTOR_DB_DIR, OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)

if not os.path.exists(ALL_RECORDS_FILE):
    from google.colab import files
    print("Chọn file all_records.json để upload...")
    uploaded = files.upload()
    src = next(iter(uploaded))
    shutil.move(src, ALL_RECORDS_FILE)

print(f"✅ all_records.json: {ALL_RECORDS_FILE}")


✅ all_records.json: /content/drive/MyDrive/nutrition_rag/data_processed/all_records.json


In [ ]:
# ── Retrieval stack ───────────────────────────────────────────
!pip install -q sentence-transformers chromadb rank_bm25
!pip install -q FlagEmbedding

# ── LLM stack (BitsAndBytes thuần, không cần Unsloth) ────────
# Pin transformers < 4.56 để tránh conflict với trl/peft trên Colab
!pip install -q "transformers>=4.43.0,<4.56.0"
!pip install -q "accelerate>=0.34.0"
!pip install -q "bitsandbytes>=0.43.0"

# ── Evaluation stack ──────────────────────────────────────────
!pip install -q ragas datasets openai
!pip install -q rouge-score nltk scikit-learn

print("✅ Cài đặt xong.")
print("⚠️  QUAN TRỌNG: Chọn Runtime → Restart runtime, rồi tiếp tục chạy từ cell tiếp theo.")


✅ Cài đặt xong.
⚠️  QUAN TRỌNG: Chọn Runtime → Restart runtime, rồi tiếp tục chạy từ cell tiếp theo.


In [ ]:
import gc, os, json, re, torch, numpy as np
from tqdm import tqdm

# ── VRAM Guard config ─────────────────────────────────────────
VRAM_CONFIG = {
    "embed_device_indexing":  "cuda",   # embed khi build index → dùng GPU
    "embed_device_inference": "cpu",    # embed lúc query → CPU để nhường VRAM cho LLM
    "reranker_device":        "cpu",    # reranker luôn CPU
    "rerank_batch_size":      8,        # forward pass từng batch 8 cặp
    "rrf_pool":               30,       # Top-N sau RRF (giảm từ 50 → 30)
    "max_rerank_candidates":  20,
    "embed_batch_size":       64,
    "max_context_chars":      2400,
    "max_doc_chars":          480,
    "collection_name":        "nutrition_v4",
}

def clear_vram():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def print_vram(label=""):
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / 1e9
        reserv = torch.cuda.memory_reserved() / 1e9
        print(f"[VRAM {label}] Alloc: {alloc:.2f}GB | Reserved: {reserv:.2f}GB")
    else:
        print(f"[VRAM {label}] No GPU detected")

print_vram("startup")


[VRAM startup] Alloc: 0.00GB | Reserved: 0.00GB


In [ ]:

# Mapping được xây dựng dựa trên toàn bộ food_name_en trong all_records.json (526 records)
# Sắp xếp từ dài → ngắn để tránh partial-match conflict (vd: "thịt bò" trước "bò")
VI_EN_FOOD_MAP = {
    # === NGŨ CỐC / TINH BỘT ===
    "gạo nếp":         "glutinous rice",
    "nếp xay":         "glutinous rice, milled",
    "gạo lứt":         "rice, brown or hulled",
    "gạo trắng":       "ordinary polished rice",
    "cơm trắng":       "ordinary polished rice",
    "gạo chưa xay":    "under milled, home-pounded rice",
    "gạo non":         "rice, unriped",
    "bột mì":          "wheat flour",
    "bột gạo nếp":     "glutinous rice flour",
    "bột gạo":         "rice ordinary flour",
    "bột ngô":         "yellow maize flour",
    "bột sắn":         "cassava flour",
    "bột khoai lang":  "sweet potato flour",
    "bột dong":        "canna edulis ker, flour",
    "bánh mì":         "french bread",
    "bánh phở":        "rice noodles",
    "bánh gạo":        "rice cake, plain",
    "bánh tráng":      "rice paper for rollers",
    "bánh cuốn":       "rice paper for rollers",
    "bún":             "rice vermicelli",
    "mì tôm":          "wheat noodles",
    "mì":              "wheat noodles",
    "miến dong":       "vermicelli from bermuda",
    "miến":            "vermicelli from bermuda",
    "ngô bắp":         "fresh maize seeds",
    "ngô khô":         "yellow maize, dried seeds",
    "ngô":             "maize",
    "bắp":             "maize",
    "khoai lang vàng": "sweet potato, yellow",
    "khoai lang trắng":"sweet potato, white",
    "khoai lang":      "sweet potato",
    "khoai tây":       "potato, white",
    "khoai mì":        "cassava",
    "sắn đắng":        "bitter cassava",
    "sắn":             "cassava",
    "khoai sọ":        "taro tuber",
    "khoai nước":      "water-taro",
    "khoai từ":        "chinese yam",
    "củ từ":           "yam winged",
    "dong riềng":      "canna edulis",
    "củ năng":         "water-caltrops",
    "sen":             "lotus",
    "hạt sen":         "lotus seed",
    "ngó sen":         "lotus, stem",
    "măng":            "bamboo shoot",
    # === ĐẬU / HẠT ===
    "đậu đen":         "black bean seeds",
    "đậu xanh":        "mungo bean seeds",
    "giá đỗ xanh":     "mungobean sprouts",
    "giá đỗ":          "mungobean sprouts",
    "giá":             "mungobean sprouts",
    "đậu phộng rang":  "cashew, common, roasted",
    "đậu phộng":       "peanut",
    "lạc rang":        "peanut, oil fried",
    "lạc":             "peanut",
    "bột lạc":         "peanut flour",
    "đậu tương":       "soybean",
    "đậu nành":        "soybean",
    "đậu hũ chiên":    "curd tofu, fried",
    "đậu hũ":          "curd tofu",
    "đậu phụ chiên":   "curd tofu, fried",
    "đậu phụ":         "curd tofu",
    "sữa đậu nành":    "soybean milk",
    "đậu đỏ":          "kidney bean",
    "đậu hà lan":      "peas garden",
    "đậu đũa":         "cow-peas",
    "đậu cove":        "french bean",
    "hạt dẻ":          "chestnut, chinese",
    "hạt điều":        "cashew nut",
    "hạt mè":          "sesame oriental seeds",
    "vừng":            "sesame oriental seeds",
    "hạt bí":          "pumpkin seeds",
    "hạt dưa hấu":     "water melon seeds",
    # === NẤM / TẢNG BIỂN ===
    "nấm hương khô":   "mushroom chinese, dried",
    "nấm hương":       "mushroom, chinese, raw",
    "nấm rơm":         "mushroom, straw",
    "nấm mèo khô":     "jew's ear, dried",
    "nấm mèo":         "jew's ear",
    "nấm thường":      "mushroom, common",
    "rong biển khô":   "dried seaweed",
    "rong biển":       "seaweed fresh",
    # === RAU CỦ ===
    "bí đỏ lá":        "pumpkin leaves",
    "bí đỏ":           "pumpkin squash",
    "bí xanh":         "asgourd waxgoured",
    "bí đao":          "asgourd waxgoured",
    "bầu":             "calabash, bottle gourd",
    "mướp đắng":       "balsam-pear",
    "khổ qua":         "balsam-pear",
    "mướp":            "gourd, sponge gourd",
    "su su":           "chayote",
    "cà tím nhỏ":      "egg plant - small",
    "cà tím":          "egg plant big",
    "cà chua":         "tomato",
    "cà rốt khô":      "dried carrot",
    "cà rốt":          "carrots",
    "bắp cải đỏ":      "cabbage, red",
    "bắp cải":         "cabbage, common",
    "cải thảo":        "chinese cabbage, white",
    "cải xoăn":        "cauliflower, green",
    "súp lơ trắng":    "cauliflower, white",
    "súp lơ xanh":     "cauliflower, green",
    "súp lơ":          "cauliflower",
    "cải xanh dưa":    "mustard green, pickled",
    "cải xanh":        "mustard greens",
    "rau muống khô":   "water spinach, dried",
    "rau muống":       "swamp cabbage",
    "rau ngót khô":    "sauropus, dried",
    "rau ngót":        "sauropus",
    "rau má":          "wort, india penny",
    "rau đay":         "jute potherb",
    "rau dền trắng":   "amaranth, sp white",
    "rau dền đỏ":      "amaranth, sp. red",
    "rau dền":         "amaranth",
    "rau cần":         "celery, chinese",
    "xà lách":         "lettuce garden",
    "rau mùi":         "coriander",
    "rau thơm":        "coriander",
    "húng quế":        "basil sweet leaves",
    "húng":            "mint leaves",
    "tía tô":          "perilla",
    "hành tây dưa":    "onion, pickled",
    "hành tây":        "onion, common, garden",
    "hành lá":         "onion, welsh",
    "hành phi":        "onion, fragrant",
    "tỏi":             "garlic bulbs",
    "gừng bột":        "ginger root, dried powder",
    "gừng":            "ginger root, fresh",
    "nghệ khô":        "turmeric rhizome, dried powder",
    "nghệ":            "turmeric, rhizome, fresh",
    "ớt đỏ":           "chili pepper",
    "ớt bột":          "red pepper, dried powder",
    "ớt xanh":         "peppers, green",
    "ớt vàng":         "peppers, yellow",
    "dưa chuột dưa":   "cucumber, pickled",
    "dưa leo":         "cucumber",
    "dưa chuột":       "cucumber",
    "cà chua dưa":     "tomato, pickled",
    "củ cải trắng":    "radish garden white",
    "củ cải đỏ":       "red radish oriental",
    "củ cải khô":      "dried radish, white",
    "su hào":          "kohlrabi",
    "su hào khô":      "dried kohlrabi",
    "măng tây trắng":  "asparagus, white",
    "bí bắp cải":      "cabbage chinese, pickled",
    "rau ngổ":         "limnophila aromatic",
    "diếp cá":         "polygonum odoratum",
    "lá lốt":          "lolot",
    # === TRÁI CÂY ===
    "chuối chín":      "banana",
    "chuối xanh":      "banana common varieties, unripe",
    "chuối":           "banana",
    "cam":             "orange",
    "nước cam":        "orange juice",
    "quýt":            "tangerine",
    "mandarin":        "mandarin",
    "bưởi":            "pomelo, pummelo",
    "chanh":           "lemon",
    "dứa dại":         "pineapple, wild",
    "dứa":             "pineapple",
    "thơm":            "pineapple",
    "xoài chín":       "mango, common; india mango, ripe",
    "xoài xanh":       "mango, common; indian mango, unripe",
    "xoài":            "mango",
    "ổi":              "guava, common",
    "đu đủ chín":      "papaya, ripe",
    "đu đủ xanh":      "papaya, unripe",
    "đu đủ":           "papaya",
    "dưa hấu":         "watermelon",
    "dưa lưới":        "musk melon",
    "dưa vàng":        "honey dew melon",
    "nho":             "grape",
    "nhãn":            "longan",
    "nhãn khô":        "longan, dried",
    "vải":             "litchi",
    "vải khô":         "litchi, dried",
    "chôm chôm":       "rambutan",
    "mận":             "japanese, plum",
    "đào":             "peach",
    "lê":              "pear",
    "táo":             "apple, common, domestic",
    "mít":             "jackfruit",
    "sầu riêng":       "durian",
    "bơ tươi":         "avocado, green",
    "bơ tím":          "avocado, purple",
    "bơ":              "avocado",
    "thanh long":      "dragon's eyes fruit",
    "khế":             "carambola",
    "na":              "sugarapple, sweetsop",
    "hồng":            "persimmon kaki",
    "me":              "tamarind fruit",
    "gấc":             "gac fruit",
    "kiwi":            "kiwi fruit",
    "dâu tây":         "strawberry",
    "dâu đen":         "blackberry",
    "mơ khô":          "apricot dried",
    "táo tàu":         "jujube, common or chinese",
    # === THỊT ===
    "thịt bò loại i":   "beef, grade i",
    "thịt bò loại ii":  "beef, grade ii",
    "thịt bò loại 1":   "beef, grade i",
    "thịt bò loại 2":   "beef, grade ii",
    "thịt bò sấy":      "dried beef",
    "thịt bò hộp":      "beef, canned",
    "thịt bò":          "beef",
    "bò":               "beef",
    "thịt lợn nạc mỡ": "pork, lean and fat",
    "thịt lợn nạc":     "pork, lean",
    "thịt lợn mỡ":      "pork, medium fat",
    "thịt lợn sườn":    "pork, ribs",
    "thịt lợn đùi":     "pork, leg",
    "thịt lợn hộp":     "pork, canned",
    "thịt lợn":         "pork",
    "thịt heo nạc":     "pork, lean",
    "thịt heo":         "pork",
    "heo":              "pork",
    "lợn":              "pork",
    "giăm bông":        "ham, pork",
    "xúc xích lợn":     "pork sausage",
    "xúc xích trung quốc": "chinese sausage",
    "da lợn":           "pork skin",
    "thịt gà ta":       "chicken meat, average",
    "thịt gà hộp":      "chicken, canned",
    "thịt gà":          "chicken",
    "gà":               "chicken",
    "thịt vịt":         "duck meat, average",
    "vịt":              "duck meat",
    "thịt ngỗng":       "goose",
    "thịt dê":          "goat, meat, lean",
    "dê":               "goat",
    "thịt cừu":         "mutton meat, lean",
    "thịt trâu nạc":    "buffalo meat, lean",
    "thịt trâu khô":    "buffalo meat, dried",
    "thịt trâu":        "buffalo meat, average",
    "trâu":             "buffalo meat",
    "thịt bê nạc mỡ":   "veal meat, lean and fat",
    "thịt bê":          "veal meat",
    "bê":               "veal",
    "thịt ngựa":        "horse meat",
    "thịt thỏ":         "rabbit meat",
    "thịt chó đùi":     "dog, shoulder",
    "thịt chó":         "dog meat",
    "thịt nai":         "deer meat",
    "thịt gà tây":      "turkey raw flesh",
    "thịt chim bồ câu": "pigeon young bird",
    # === PHỦ TẠNG ===
    "gan bò":           "beef, liver",
    "tim bò":           "beef, heart",
    "thận bò":          "beef, kidney",
    "phổi bò":          "beef, lung",
    "lưỡi bò":          "beef tongue",
    "dạ dày bò":        "stomach, beef",
    "óc bò":            "brain beef",
    "tiết bò":          "beef, blood",
    "đuôi bò":          "beef, tail",
    "tủy bò":           "beef bone marrow",
    "đầu bò":           "head beef",
    "gan lợn":          "pork liver",
    "gan heo":          "pork liver",
    "tim lợn":          "hog heart",
    "thận lợn":         "pork, kidney",
    "phổi lợn":         "hog lung",
    "lòng lợn":         "hog intestine",
    "dạ dày lợn":       "stomach, hog",
    "óc lợn":           "hog brain",
    "tiết lợn":         "hog blood",
    "tai lợn":          "hog ears",
    "đầu lợn":          "hog, head",
    "lưỡi lợn":         "hog, tongue",
    "đuôi lợn":         "hog, tail",
    "tủy lợn":          "hog, bone marrow",
    "mề gà":            "chicken gizzard",
    "tim gà":           "chicken heart",
    "gan gà":           "chicken liver",
    "lòng gà":          "chicken giblets",
    "gan vịt":          "duck liver",
    # === CÁ / HẢI SẢN ===
    "cá chép":          "carp",
    "cá trắm":          "carp, amur",
    "cá rô phi":        "tilapia",
    "cá trôi":          "major carp",
    "cá rô":            "anabas",
    "cá lóc":           "fish, snake head",
    "cá quả":           "fish, snake head",
    "cá hồi":           "salmon",
    "cá thu":           "mackerel",
    "cá ngừ":           "flying fish, tuna",
    "cá mòi":           "sardin",
    "cá trích":         "herring",
    "cá chình":         "eel",
    "cá da trơn":       "catfish",
    "cá bơn":           "flounder",
    "cá đối":           "mullet",
    "cá bống":          "goby",
    "cá cơm":           "scad, anchovy",
    "cá khô":           "dried fish",
    "cá mắm":           "shredded snake-head fish, salted",
    "tôm biển":         "sea shrimp",
    "tôm khô":          "shrimp, dried",
    "tôm moi":          "tiny shrimp",
    "tôm tép":          "fresh-water shrimp",
    "tôm":              "shrimp",
    "cua biển":         "crab, sea water",
    "cua đồng":         "crab, fresh water",
    "ghẹ":              "small sea - crab",
    "mực khô":          "dried cuttle fish",
    "mực":              "cuttle fish, raw",
    "bạch tuộc":        "cuttle fish",
    "nghêu":            "clam",
    "hàu":              "oyster",
    "ốc sên":           "snail large",
    "ốc":               "snail medium",
    "sứa":              "sea slug",
    "ếch":              "frog",
    "tằm":              "silk worm",
    "châu chấu":        "locust",
    # === TRỨNG ===
    "trứng gà bột":     "chicken egg powder",
    "trứng gà lòng trắng": "hen egg, white",
    "trứng gà lòng đỏ":  "hen egg, yolk",
    "trứng gà":         "hen egg, raw, whole",
    "trứng vịt lộn":    "duck egg, embryonated",
    "trứng vịt lòng trắng": "duck egg, white",
    "trứng vịt lòng đỏ": "duck egg, yolk",
    "trứng vịt":        "duck egg, whole",
    "trứng cút":        "quail egg",
    # === SỮA ===
    "sữa mẹ":           "breast milk",
    "sữa bột nguyên kem": "milk powder, whole",
    "sữa bột tách béo": "skimmed milk powder",
    "sữa đặc có đường": "milk condensed, sweetened",
    "sữa bột đậu":      "milk flour, made from roasted soybeans",
    "sữa bột":          "milk powder",
    "sữa chua sôcôla":  "yogurt, chocolate",
    "sữa chua":         "yogurt",
    "sữa tươi bò":      "milk cow, fresh",
    "sữa bò":           "milk cow",
    "sữa tươi":         "milk cow, fresh",
    "sữa dê":           "milk goat's",
    "phô mai":          "cheese",
    # === DẦU MỠ ===
    "bơ động vật":      "butter, unsalted",
    "bơ thực vật":      "butter-margarine blend",
    "mỡ lợn lỏng":      "lard, liquid",
    "mỡ lợn":           "lard",
    "dầu dừa":          "coconut oil",
    "dầu mè":           "sesame oil",
    "dầu đậu phộng":    "peanut oil",
    "dầu đậu nành":     "soybean oil",
    "dầu ngô":          "corn oil",
    "dầu ô liu":        "olive oil",
    "dầu cọ":           "palm oil",
    "dầu cám":          "rice bran oil",
    "dầu hạt bông":     "cottonseed oil",
    "dầu thực vật":     "vegetable oil",
    "sốt mayonnaise":   "mayonnaise",
    # === GIA VỊ ===
    "đường trắng":      "refined sugar",
    "mật ong":          "honey",
    "muối":             "table salt",
    "tiêu":             "peppercorn",
    "ớt khô bột":       "red pepper, dried powder",
    "nước mắm thượng hạng": "fish - sauce (super quality)",
    "nước mắm hạng 1":  "fish sauce, grade i",
    "nước mắm hạng 2":  "fish sauce, grade ii",
    "nước mắm cô":      "fish sauce, concentrated",
    "nước mắm":         "fish sauce",
    "tương đậu":        "soybean sauce",
    "tương nếp":        "soybean sauce with glutinous rice",
    "mắm tôm chiên":    "deep fried shrimp paste",
    "mắm tôm":          "shrimp paste",
    "mắm ruốc":         "shrimp sauce",
    "cà ri":            "cari powder",
    # === ĐỒ UỐNG ===
    "cà phê":           "coffee",
    "ca cao":           "cocoa powder",
    "bia":              "beer light",
    "rượu vang đỏ":     "red wine",
    "rượu vang trắng":  "white wine",
    "rượu vang":        "red wine",
    "nước dừa":         "coconut milk",
    "nước cam tươi":    "orange juice, fresh",
    "nước cà chua":     "tomato juice",
    "coca cola":        "coca cola",
    "vodka":            "vodka",
    "whisky":           "whisky liqueur",
}

# Sort by length desc để tránh partial-match (khớp "thịt bò loại i" trước "thịt bò")
_SORTED_MAP = sorted(VI_EN_FOOD_MAP.items(), key=lambda x: len(x[0]), reverse=True)

def translate_vi_food_terms(text: str) -> str:
    result = text.lower()
    for vi_term, en_term in _SORTED_MAP:
        if vi_term in result:
            result = result.replace(vi_term, en_term)
    return result

def detect_language(text: str) -> str:
    vn_chars = r'[àáảãạăắằẳẵặâấầẩẫậèéẻẽẹêếềểễệìíỉĩịòóỏõọôốồổỗộơớờởỡợùúủũụưứừửữựỳýỷỹỵđ]'
    return "vi" if re.search(vn_chars, text, re.IGNORECASE) else "en"

# ── Smoke test ────────────────────────────────────────────────
tests = [
    "thịt bò loại i chứa bao nhiêu protein",
    "gan lợn nhiều sắt không",
    "trứng gà bao nhiêu calo",
    "cá hồi và cá thu loại nào nhiều omega không",
]
print("=== Translation smoke test ===")
for t in tests:
    print(f"  VI: {t}")
    print(f"  EN: {translate_vi_food_terms(t)}")
    print()


=== Translation smoke test ===
  VI: thịt bò loại i chứa bao nhiêu protein
  EN: beef, grade i chứa bao nhiêu protein

  VI: gan lợn nhiều sắt không
  EN: pork liver nhiều sắt không

  VI: trứng gà bao nhiêu calo
  EN: hen egg, raw, whole bao nhiêu calo

  VI: cá hồi và cá thu loại nào nhiều omega không
  EN: salmon và mackerel loại nào nhiều otamarind fruitga không



In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

# ── Load records ──────────────────────────────────────────────
with open(ALL_RECORDS_FILE, "r", encoding="utf-8") as f:
    all_records = json.load(f)
print(f"Loaded {len(all_records)} records")

ids       = [r["record_id"]          for r in all_records]
documents = [r["text_for_embedding"] for r in all_records]

# ── Enrich documents: thêm tên Việt để BM25 khớp được VI queries ──
def enrich_text(record: dict) -> str:
    """
    Gắn tên Việt vào đầu text_for_embedding.
    BM25 tokenize sẽ tìm thấy 'thịt bò' trong corpus thay vì chỉ 'beef'.
    """
    base = record["text_for_embedding"]
    en = str(record.get("food_name_en", "")).lower()
    # Tìm tên Việt tương ứng qua reverse lookup
    vi_names = [vi for vi, en_term in VI_EN_FOOD_MAP.items()
                if en_term.lower() in en and len(vi) > 3]
    # Giữ tên Việt ngắn nhất (generic nhất)
    vi_names = sorted(vi_names, key=len)[:3]
    if vi_names:
        vi_label = " / ".join(vi_names)
        return f"Tên tiếng Việt: {vi_label}\n{base}"
    return base

enriched_documents = [enrich_text(r) for r in all_records]
print(f"Sample enriched doc:\n{enriched_documents[0][:200]}")

# ── Embedding model: BAAI/bge-m3 ─────────────────────────────
print("\nĐang tải BAAI/bge-m3 (sẽ dùng GPU khi build index, CPU khi query)...")
embed_model = SentenceTransformer(
    "BAAI/bge-m3",
    device=VRAM_CONFIG["embed_device_indexing"]
)
print_vram("after bge-m3 load")

# ── ChromaDB ──────────────────────────────────────────────────
chroma_client = chromadb.PersistentClient(path=VECTOR_DB_DIR)
collection = chroma_client.get_or_create_collection(
    name=VRAM_CONFIG["collection_name"],
    metadata={"hnsw:space": "cosine"}
)

if collection.count() > 0:
    print(f"ChromaDB: {collection.count()} records đã có → skip re-embed")
else:
    print("ChromaDB trống → bắt đầu embedding...")
    metadatas = []
    for r in all_records:
        m = {"record_type": r.get("record_type",""), "source": r.get("source","")}
        for k in ("group","age","food_name_en"):
            if r.get(k): m[k] = str(r[k])
        metadatas.append(m)

    BS = VRAM_CONFIG["embed_batch_size"]
    for i in tqdm(range(0, len(ids), BS)):
        s, e = i, i+BS
        embs = embed_model.encode(enriched_documents[s:e], convert_to_tensor=False).tolist()
        collection.upsert(ids=ids[s:e], documents=enriched_documents[s:e],
                          embeddings=embs, metadatas=metadatas[s:e])
    print(f"✅ {collection.count()} vectors → ChromaDB")

# Sau khi build index: chuyển embed_model sang CPU để nhường VRAM cho LLM
embed_model = SentenceTransformer("BAAI/bge-m3", device=VRAM_CONFIG["embed_device_inference"])
clear_vram()
print_vram("after moving embed to CPU")

# ── BM25 trên enriched corpus ─────────────────────────────────
print("Building BM25...")
tokenized_corpus = [doc.lower().split() for doc in enriched_documents]
bm25 = BM25Okapi(tokenized_corpus)
print("✅ BM25 ready")


Loaded 583 records
Sample enriched doc:
Tên tiếng Việt: gạo nếp / nếp xay
Food name: Glutinous rice, milled
Calories: 344.0 kcal
Protein: 8.6 g
Fat: 1.5 g
Carbohydrate: 74.5 g
Fiber: 0.6 g
Calcium: 32.0 mg
Iron: 1.2 mg

Đang tải BAAI/bge-m3 (sẽ dùng GPU khi build index, CPU khi query)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


[VRAM after bge-m3 load] Alloc: 2.27GB | Reserved: 2.29GB
ChromaDB: 583 records đã có → skip re-embed
[VRAM after moving embed to CPU] Alloc: 0.00GB | Reserved: 0.00GB
Building BM25...
✅ BM25 ready


In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
import torch

MODEL_ID = "NousResearch/Meta-Llama-3-8B-Instruct"

print(f"Đang tải {MODEL_ID} (4-bit NF4 BitsAndBytes)...")
print_vram("before LLM load")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
model.eval()

print("✅ Llama-3 sẵn sàng.")
print_vram("after LLM load")


def llm_call(
    system_prompt: str,
    user_prompt: str,
    max_new_tokens: int = 256,
    temperature: float = 0.1,
) -> str:
    """Gọi Llama-3 với temperature thấp, trả về text đã strip."""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to("cuda")

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=temperature > 0,
            pad_token_id=tokenizer.eos_token_id,
        )

    gen_ids = out[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(gen_ids, skip_special_tokens=True).strip()


# Smoke test
resp = llm_call(
    "You are helpful.",
    "Say hello in one sentence.",
    max_new_tokens=30,
)

print(f"Smoke test: {resp}")

Đang tải NousResearch/Meta-Llama-3-8B-Instruct (4-bit NF4 BitsAndBytes)...
[VRAM before LLM load] Alloc: 0.00GB | Reserved: 0.00GB


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


✅ Llama-3 sẵn sàng.
[VRAM after LLM load] Alloc: 5.71GB | Reserved: 7.23GB
Smoke test: Hello!


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [ ]:
ROUTE_CONFIG = {
    "LOOKUP":    {"top_k": 3},
    "COMPARE":   {"top_k": 8},
    "RECOMMEND": {"top_k": 10},
    "EXPLAIN":   {"top_k": 5},
}

def route_query(query: str) -> str:
    lang = detect_language(query)
    if lang == "vi":
        system = (
            "Phân loại câu hỏi vào 1 trong 4 loại:\n"
            "- LOOKUP: tra cứu số liệu 1 thực phẩm.\n"
            "- COMPARE: so sánh 2+ thực phẩm.\n"
            "- RECOMMEND: gợi ý thực đơn.\n"
            "- EXPLAIN: giải thích khái niệm.\n"
            "Chỉ trả về 1 từ duy nhất."
        )
    else:
        system = (
            "Classify the question into one of: LOOKUP, COMPARE, RECOMMEND, EXPLAIN.\n"
            "Return only the one-word label."
        )
    raw = llm_call(system, query, max_new_tokens=8, temperature=0.0)
    for intent in ["LOOKUP","COMPARE","RECOMMEND","EXPLAIN"]:
        if intent in raw.upper():
            return intent
    return "LOOKUP"

print(route_query("Thịt bò grade I có bao nhiêu protein?"))
print(route_query("So sánh cá hồi và cá thu"))


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


LOOKUP
COMPARE


In [ ]:
import numpy as np

def decompose_query(query: str) -> list:
    lang = detect_language(query)

    if lang == "vi":
        system = (
            "Nhiệm vụ: Tách câu hỏi so sánh thành các câu hỏi tra cứu độc lập.\n"
            "KHÔNG trả lời câu hỏi. CHỈ tách thành các câu hỏi nhỏ hơn.\n"
            "Mỗi câu hỏi con trên 1 dòng, bắt đầu bằng dấu '-'.\n"
            "Nếu câu hỏi đơn giản (chỉ 1 thực phẩm), trả về đúng câu gốc.\n\n"
            "Ví dụ:\n"
            "Input: Thịt bò và thịt heo loại nào nhiều đạm hơn?\n"
            "Output:\n"
            "- Thịt bò có bao nhiêu gam đạm (protein) trên 100g?\n"
            "- Thịt heo có bao nhiêu gam đạm (protein) trên 100g?\n\n"
            "Input: Cơm trắng 100g có bao nhiêu calo?\n"
            "Output:\n"
            "- Cơm trắng 100g có bao nhiêu calo?\n"
        )
    else:
        system = (
            "Task: Split comparison questions into independent lookup sub-questions.\n"
            "Do NOT answer the question. ONLY split into smaller questions.\n"
            "Each sub-question on its own line starting with '-'.\n"
            "If the question is simple (only 1 food), return the original question.\n\n"
            "Example:\n"
            "Input: Which has more protein, beef or pork?\n"
            "Output:\n"
            "- How many grams of protein per 100g does beef have?\n"
            "- How many grams of protein per 100g does pork have?\n"
        )

    raw = llm_call(system, query, max_new_tokens=120, temperature=0.0)

    subs = [
        l.strip("- •").strip()
        for l in raw.splitlines()
        if len(l.strip()) > 5
        and not any(keyword in l.lower() for keyword in ["output:", "input:", "ví dụ", "example"])
    ][:5]

    return subs if subs else [query]


def generate_hyde_doc(query_en: str) -> str:
    """Luôn sinh bằng tiếng Anh để match English corpus."""

    system = (
        "You are a nutrition expert. Write ~50 words in English describing "
        "the typical nutritional values (calories, protein, fat, carbs) of the food asked about. "
        "Only write the description."
    )

    return llm_call(system, query_en, max_new_tokens=90, temperature=0.1)


def get_query_embedding(text: str) -> list:
    """Encode trên CPU (embed_model đã chuyển sang CPU sau khi build index)."""
    return embed_model.encode(text, convert_to_tensor=False).tolist()


def get_hyde_embedding(query_en: str, use_hyde: bool = True):
    q_emb = np.array(get_query_embedding(query_en))

    if use_hyde:
        hyde = generate_hyde_doc(query_en)
        h_emb = np.array(get_query_embedding(hyde))
        return ((q_emb + h_emb) / 2.0).tolist(), hyde

    return q_emb.tolist(), None


# Test
subs = decompose_query("So sánh lượng đạm trong thịt bò loại I và ức gà, và ghi rõ từng nguồn dinh dưỡng của tụi nó ra")
print("Decompose:", subs)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Decompose: ['Thịt bò loại I có bao nhiêu gam đạm (protein) trên 100g?', 'Uc gà có bao nhiêu gam đạm (protein) trên 100g?', 'Nguồn dinh dưỡng của thịt bò loại I là gì?', 'Nguồn dinh dưỡng của ức gà là gì?']


In [ ]:
from FlagEmbedding import FlagReranker

print("Đang tải bge-reranker-base (CPU)...")
reranker = FlagReranker("BAAI/bge-reranker-base",
                         use_fp16=False,
                         device=VRAM_CONFIG["reranker_device"])
print("✅ Reranker ready")
print_vram("after reranker")

def reciprocal_rank_fusion(dense_ids, sparse_ids, k=60):
    scores = {}
    for rank, did in enumerate(dense_ids):
        scores[did] = scores.get(did, 0) + 1/(k+rank+1)
    for rank, did in enumerate(sparse_ids):
        scores[did] = scores.get(did, 0) + 1/(k+rank+1)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)

def rerank_documents(query: str, candidate_ids: list, top_n: int = 5) -> list:
    if not candidate_ids:
        return []
    cands = candidate_ids[:VRAM_CONFIG["max_rerank_candidates"]]
    res   = collection.get(ids=cands)
    id2txt = {res["ids"][i]: res["documents"][i] for i in range(len(res["ids"]))}

    all_scores = []
    bs = VRAM_CONFIG["rerank_batch_size"]
    valid_ids = [d for d in cands if d in id2txt]
    for i in range(0, len(valid_ids), bs):
        batch = valid_ids[i:i+bs]
        pairs = [(query, id2txt[d][:VRAM_CONFIG["max_doc_chars"]]) for d in batch]
        all_scores.extend(zip(batch, reranker.compute_score(pairs)))
    all_scores.sort(key=lambda x: x[1], reverse=True)
    return [did for did, _ in all_scores[:top_n]]

def hybrid_search(query: str, top_k: int = 5,
                  use_hyde: bool = True, rerank: bool = True) -> str:
    """
    Full pipeline: VI→EN translate → Dense(HyDE) + Sparse(BM25-EN) → RRF → Rerank
    """
    lang    = detect_language(query)
    en_query = translate_vi_food_terms(query) if lang == "vi" else query

    # 1. Dense (HyDE, embedding CPU)
    q_emb, _ = get_hyde_embedding(en_query, use_hyde=use_hyde)
    dense_res = collection.query(
        query_embeddings=[q_emb],
        n_results=min(VRAM_CONFIG["rrf_pool"], collection.count())
    )
    dense_ids = dense_res["ids"][0]

    # 2. Sparse BM25 dùng EN query (corpus đã enriched)
    bm25_scores = bm25.get_scores(en_query.lower().split())
    sparse_ids  = [ids[i] for i in np.argsort(bm25_scores)[::-1][:VRAM_CONFIG["rrf_pool"]]]

    # 3. RRF
    rrf = reciprocal_rank_fusion(dense_ids, sparse_ids)
    cands = [d for d, _ in rrf[:VRAM_CONFIG["rrf_pool"]]]

    # 4. Rerank
    top_ids = rerank_documents(query, cands, top_n=top_k) if rerank else cands[:top_k]

    if not top_ids:
        return ""
    data = collection.get(ids=top_ids)
    id2doc = {data["ids"][i]: (data["documents"][i], data["metadatas"][i])
              for i in range(len(data["ids"]))}
    parts = []
    for i, did in enumerate(top_ids, 1):
        if did in id2doc:
            txt, meta = id2doc[did]
            parts.append(
                f"--- Doc {i} (ID:{did}|Src:{meta.get('source','?')}) ---\n"
                + txt[:VRAM_CONFIG["max_doc_chars"]]
            )
    return "\n\n".join(parts)

# Test
ctx = hybrid_search("Thịt bò loại I chứa bao nhiêu protein?", top_k=3)
print(ctx[:600])


Đang tải bge-reranker-base (CPU)...
✅ Reranker ready
[VRAM after reranker] Alloc: 5.71GB | Reserved: 7.28GB


You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


--- Doc 1 (ID:food_0282|Src:food_composition_vn_2007) ---
Tên tiếng Việt: thịt bò / thịt bò loại i / thịt bò loại 1
Food name: Beef, grade II
Calories: 167.0 kcal
Protein: 18.0 g
Fat: 10.5 g
Carbohydrate: 0.0 g
Fiber: 0.0 g
Calcium: 10.0 mg
Iron: 2.7 mg

--- Doc 2 (ID:food_0281|Src:food_composition_vn_2007) ---
Tên tiếng Việt: thịt bò / thịt bò loại i / thịt bò loại 1
Food name: Beef, grade I
Calories: 118.0 kcal
Protein: 21.0 g
Fat: 3.8 g
Carbohydrate: 0.0 g
Fiber: 0.0 g
Calcium: 12.0 mg
Iron: 3.1 mg

--- Doc 3 (ID:food_0283|Src:food_composition_vn_2007) ---
Tên tiếng Việt: thịt bò
Food name:


In [ ]:
STRICT_SYSTEM_VI = """Bạn là chuyên gia dinh dưỡng AI. Chỉ được dùng thông tin từ [CONTEXT].
QUY TẮC:
1. TUYỆT ĐỐI không tự bịa hoặc suy diễn số liệu ngoài [CONTEXT].
2. Mọi con số (kcal, protein...) PHẢI trích từ [CONTEXT] kèm đơn vị.
3. Nếu [CONTEXT] không có thông tin → trả lời: "Không có thông tin trong tài liệu."
4. Trả lời bằng tiếng Việt, ngắn gọn.
5. Nếu thực phẩm hỏi là dạng chín (cơm, cháo...) nhưng [CONTEXT] chỉ có dạng khô/sống, hãy ghi rõ: "(lưu ý: giá trị trên là dạng khô/sống, cơm/món chín sẽ thấp hơn do hút nước)"""

STRICT_SYSTEM_EN = """You are a nutrition AI expert. Only use information from [CONTEXT].
RULES:
1. NEVER fabricate numbers outside [CONTEXT].
2. Every number (kcal, protein...) MUST come from [CONTEXT] with units.
3. If [CONTEXT] lacks the info → reply: "No information available in the documents."
4. Reply in English, concise.
5. If the food asked is cooked (rice, porridge...) but [CONTEXT] is only available in dry/raw form, please clearly state: "(note: the above value is dry/raw form, cooked rice/dish will be lower due to water absorption)"""

def full_rag_answer(query: str, verbose: bool = True) -> str:
    if verbose: print(f"\n👤 {query}")

    lang    = detect_language(query)
    intent  = route_query(query)
    top_k   = ROUTE_CONFIG.get(intent, {"top_k":4})["top_k"]
    if verbose: print(f"⚙️  Intent:{intent} | Lang:{lang} | top_k:{top_k}")


    sub_queries = decompose_query(query)

    if verbose and len(sub_queries)>1:
        print(f"🔀 Decomposed: {sub_queries}")

    contexts = []
    for sq in sub_queries:
        use_hyde = intent != "LOOKUP"
        ctx = hybrid_search(sq, top_k=top_k, use_hyde=use_hyde, rerank=True)
        if ctx: contexts.append(ctx)

    combined = "\n\n".join(contexts)
    if not combined:
        return "Không tìm thấy thông tin phù hợp." if lang=="vi" else "No relevant information found."

    combined = combined[:VRAM_CONFIG["max_context_chars"]]
    system   = STRICT_SYSTEM_VI if lang=="vi" else STRICT_SYSTEM_EN
    prompt   = f"[CONTEXT]\n{combined}\n\n[CÂU HỎI]\n{query}\n\n[CÂU TRẢ LỜI]"
    answer   = llm_call(system, prompt, max_new_tokens=300, temperature=0.1)

    if verbose: print(f"🤖 {answer}\n{'='*70}")
    return answer

# Demo
full_rag_answer("Thịt bò và thịt heo cái loại nào nhiều đạm hơn?")
full_rag_answer("Cơm trắng 100g có bao nhiêu calo?")


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



👤 Thịt bò và thịt heo cái loại nào nhiều đạm hơn?


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


⚙️  Intent:COMPARE | Lang:vi | top_k:8
🔀 Decomposed: ['Thịt bò có bao nhiêu gam đạm (protein) trên 100g?', 'Thịt heo có bao nhiêu gam đạm (protein) trên 100g?']


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


🤖 Thịt bò và thịt heo có nhiều loại khác nhau, nhưng xét về mặt trung bình, thịt bò có hàm lượng protein cao hơn. Theo các tài liệu, thịt bò có hàm lượng protein trung bình là 18,0 g (từ 14,8 g đến 23,1 g), trong khi thịt heo có hàm lượng protein trung bình là 16,2 g (từ 16,2 g đến 21,7 g). Do đó, thịt bò có nhiều đạm hơn.

👤 Cơm trắng 100g có bao nhiêu calo?


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


⚙️  Intent:LOOKUP | Lang:vi | top_k:3
🤖 Cơm trắng 100g có 344 kcal. (lưu ý: giá trị trên là dạng khô, cơm chín sẽ thấp hơn do hút nước)


'Cơm trắng 100g có 344 kcal. (lưu ý: giá trị trên là dạng khô, cơm chín sẽ thấp hơn do hút nước)'

In [ ]:
from sklearn.metrics import classification_report

router_tests = [
    ("Cơm trắng 100g có bao nhiêu calo?",                "LOOKUP"),
    ("Hàm lượng vitamin C trong cam là bao nhiêu?",      "LOOKUP"),
    ("Thịt bò và thịt gà loại nào nhiều protein hơn?",   "COMPARE"),
    ("Sữa tươi nguyên kem so với sữa ít béo khác gì?",   "COMPARE"),
    ("Tôi muốn giảm cân, ăn gì cho bữa sáng?",           "RECOMMEND"),
    ("Thực đơn 7 ngày cho người tiểu đường type 2",       "RECOMMEND"),
    ("Chỉ số GI là gì và ảnh hưởng thế nào đến đường huyết?","EXPLAIN"),
    ("Tại sao thiếu sắt lại gây thiếu máu?",              "EXPLAIN"),
]
y_true = [t for _, t in router_tests]
y_pred = [route_query(q) for q, _ in router_tests]
print(classification_report(y_true, y_pred,
      labels=["LOOKUP","COMPARE","RECOMMEND","EXPLAIN"], zero_division=0))


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.

              precision    recall  f1-score   support

      LOOKUP       1.00      1.00      1.00         2
     COMPARE       1.00      1.00      1.00         2
   RECOMMEND       1.00      1.00      1.00         2
     EXPLAIN       1.00      1.00      1.00         2

    accuracy                           1.00         8
   macro avg       1.00      1.00      1.00         8
weighted avg       1.00      1.00      1.00         8



In [ ]:
def recall_at_k(retrieved, relevant, k):
    return len(set(retrieved[:k]) & set(relevant)) / len(relevant) if relevant else 0.0

def mrr(retrieved, relevant):
    for rank, did in enumerate(retrieved, 1):
        if did in set(relevant): return 1.0/rank
    return 0.0

def get_food_id_by_name(keyword):
    kw = keyword.lower()
    for r in all_records:
        if r["record_type"]=="food" and kw in str(r.get("food_name_en","")).lower():
            return r["record_id"]
    return None

retrieval_tests = [
    {"query": "Thịt bò (Beef, grade I) chứa bao nhiêu protein?",
     "relevant_ids": [get_food_id_by_name("beef, grade i")]},
    {"query": "Gạo nếp xay có bao nhiêu calo?",
     "relevant_ids": [get_food_id_by_name("glutinous rice, milled")]},
    {"query": "Trứng gà nguyên quả bao nhiêu calo?",
     "relevant_ids": [get_food_id_by_name("hen egg, raw, whole")]},
    {"query": "Cá hồi chứa bao nhiêu protein?",
     "relevant_ids": [get_food_id_by_name("salmon")]},
]
for tc in retrieval_tests:
    tc["relevant_ids"] = [i for i in tc["relevant_ids"] if i]

metrics = {"R@1":[],"R@3":[],"R@5":[],"MRR":[]}
for tc in retrieval_tests:
    if not tc["relevant_ids"]: continue
    q   = tc["query"]
    rel = tc["relevant_ids"]
    lang = detect_language(q)
    en_q = translate_vi_food_terms(q) if lang=="vi" else q
    q_emb, _ = get_hyde_embedding(en_q, use_hyde=False)
    dense_res = collection.query(query_embeddings=[q_emb], n_results=30)
    dense_ids = dense_res["ids"][0]
    bm25_scores = bm25.get_scores(en_q.lower().split())
    sparse_ids  = [ids[i] for i in np.argsort(bm25_scores)[::-1][:30]]
    rrf_r = reciprocal_rank_fusion(dense_ids, sparse_ids)
    cands = [d for d,_ in rrf_r[:30]]
    top10 = rerank_documents(q, cands, top_n=10)
    metrics["R@1"].append(recall_at_k(top10, rel, 1))
    metrics["R@3"].append(recall_at_k(top10, rel, 3))
    metrics["R@5"].append(recall_at_k(top10, rel, 5))
    metrics["MRR"].append(mrr(top10, rel))
    print(f"Q: {q[:55]}...")
    print(f"   R@1={metrics['R@1'][-1]:.2f} R@3={metrics['R@3'][-1]:.2f} R@5={metrics['R@5'][-1]:.2f} MRR={metrics['MRR'][-1]:.2f}")

retrieval_summary = {m: round(sum(v)/len(v),3) for m,v in metrics.items() if v}
print("\nAvg:", retrieval_summary)


Q: Thịt bò (Beef, grade I) chứa bao nhiêu protein?...
   R@1=1.00 R@3=1.00 R@5=1.00 MRR=1.00
Q: Gạo nếp xay có bao nhiêu calo?...
   R@1=1.00 R@3=1.00 R@5=1.00 MRR=1.00
Q: Trứng gà nguyên quả bao nhiêu calo?...
   R@1=1.00 R@3=1.00 R@5=1.00 MRR=1.00
Q: Cá hồi chứa bao nhiêu protein?...
   R@1=1.00 R@3=1.00 R@5=1.00 MRR=1.00

Avg: {'R@1': 1.0, 'R@3': 1.0, 'R@5': 1.0, 'MRR': 1.0}


In [ ]:
import re

def extract_nums(text):
    patterns = {
        "kcal":    r'(\d+(?:[.,]\d+)?)\s*kcal',
        "protein": r'(\d+(?:[.,]\d+)?)\s*g\b[^\n]*(?:protein|đạm)',
        "fat":     r'(\d+(?:[.,]\d+)?)\s*g\b[^\n]*(?:fat|béo|lipid)',
        "carbs":   r'(\d+(?:[.,]\d+)?)\s*g\b[^\n]*(?:carb|tinh bột)',
    }
    res = {}
    for k, pat in patterns.items():
        m = re.search(pat, text, re.IGNORECASE)
        if m: res[k] = float(m.group(1).replace(",","."))
    return res

def numeric_exact_match(pred, gold, tol=0.05):
    pv, gv = extract_nums(pred), extract_nums(gold)
    res = {}
    for k, gval in gv.items():
        if k in pv:
            res[k] = abs(pv[k]-gval)/gval <= tol if gval else pv[k]==0
        else:
            res[k] = False
    return res

em_tests = [
    {"query": "Gạo nếp xay (Glutinous rice, milled) chứa bao nhiêu calo và đạm?",
     "gold":  "344 kcal, 8.6 g protein"},
    {"query": "Thịt bò (Beef, grade I) có bao nhiêu chất béo?",
     "gold":  "3.8 g fat"},
]
total, correct = 0, 0
for idx, tc in enumerate(em_tests, 1):
    pred  = full_rag_answer(tc["query"], verbose=False)
    match = numeric_exact_match(pred, tc["gold"])
    print(f"Test {idx}: {tc['query'][:55]}...")
    print(f"  Pred: {pred[:100]}")
    for k,ok in match.items():
        print(f"  {k}: {'✅' if ok else '❌'}")
        total+=1; correct+=(1 if ok else 0)
em_score = correct/total if total else 0
print(f"\nExact Match: {em_score*100:.1f}%")


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Test 1: Gạo nếp xay (Glutinous rice, milled) chứa bao nhiêu cal...
  Pred: Theo tài liệu, gạo nếp xay (Glutinous rice, milled) có hai giá trị khác nhau:

* Theo Doc 1 (ID:food
  kcal: ✅
  protein: ✅


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Test 2: Thịt bò (Beef, grade I) có bao nhiêu chất béo?...
  Pred: Thịt bò (Beef, grade I) có 3,8 g chất béo.
  fat: ✅

Exact Match: 100.0%


In [ ]:
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer as rs_mod

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

rscorer = rs_mod.RougeScorer(["rougeL"], use_stemmer=False)
smoother = SmoothingFunction().method1

nlp_tests = [
    {"query": "Gạo nếp xay (Glutinous rice, milled) chứa bao nhiêu calo?",
     "reference": "Gạo nếp chứa 344 kcal trên 100g."},
    {"query": "Thịt bò (Beef, grade I) có bao nhiêu protein?",
     "reference": "Thịt bò loại I có 21.5 g protein trên 100g."},
]
rouges, bleus = [], []
for tc in nlp_tests:
    pred = full_rag_answer(tc["query"], verbose=False)
    ref  = tc["reference"]
    r    = rscorer.score(ref, pred)
    rouges.append(r["rougeL"].fmeasure)
    rt   = nltk.word_tokenize(ref.lower())
    pt   = nltk.word_tokenize(pred.lower())
    bleus.append(sentence_bleu([rt], pt, smoothing_function=smoother))
    print(f"ROUGE-L={rouges[-1]:.3f} | BLEU={bleus[-1]:.3f} | {tc['query'][:50]}")

print(f"\nAvg ROUGE-L: {sum(rouges)/len(rouges):.3f} | Avg BLEU: {sum(bleus)/len(bleus):.3f}")


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


ROUGE-L=0.220 | BLEU=0.008 | Gạo nếp xay (Glutinous rice, milled) chứa bao nhiê


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


ROUGE-L=0.640 | BLEU=0.053 | Thịt bò (Beef, grade I) có bao nhiêu protein?

Avg ROUGE-L: 0.430 | Avg BLEU: 0.031


In [ ]:
!pip install -q ragas datasets
!pip install -q langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 41.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-classic 1.0.8 requires langchain-core<2.0.0,>=1.4.4, but you have langchain-core 0.2.43 which is incompatible.
langchain-classic 1.0.8 requires langchain-text-splitters<2.0.0,>=1.1.2, but you have langchain-text-splitters 0.2.4 which is incompatible.
instructor 1.15.1 requires openai<3.0.0,>=2.0.0, but you have openai 1.109.1 which is incompatible.
langchain-google-genai 4.2.5 requires langchain-core<2.0.0,>=1.3.2, but you have langchain-core 0.2.43 which is incompatible.
langchain-huggingface 1.2.2 requires langchain-core<2.0.0,>=1.2.31, but you have langchain-core 0.2.43 which is incompatible.
langgraph-sdk 0.4.2 requires langchain-core<2,>=1.4.0

In [ ]:
import os
from google.colab import userdata
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision

# Import các module của Google GenAI thông qua LangChain
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

# --- 1. SETUP API KEY TỪ COLAB SECRETS ---
try:
    # Lấy key từ Colab Secrets (thay 'GEMINI_API_KEY' bằng tên secret của bạn nếu khác)
    os.environ["GOOGLE_API_KEY"] = userdata.get('GEMINI_API_KEY')
except userdata.SecretNotFoundError:
    print("⚠️ Lỗi: Chưa tìm thấy Secret có tên 'GEMINI_API_KEY'. Vui lòng kiểm tra lại menu Secrets.")

# --- 2. KHỞI TẠO GEMINI MODELS CHO RAGAS ---
# Sử dụng flash để tăng tốc độ đánh giá, hoặc pro nếu bạn cần judge khắt khe hơn
gemini_llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash")
gemini_embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001")

# --- 3. CHUẨN BỊ DỮ LIỆU ĐÁNH GIÁ ---
ragas_tests = [
    {"question": "Gạo nếp xay (Glutinous rice, milled) chứa bao nhiêu calo?",
     "ground_truth": "Gạo nếp chứa 344 kcal."},
    {"question": "Thịt bò (Beef, grade I) có bao nhiêu protein?",
     "ground_truth": "Thịt bò loại I có 21.5 g protein."},
]

questions, answers, contexts_list, ground_truths = [], [], [], []

# Xử lý RAG pipeline của bạn (đoạn này giữ nguyên logic cũ)
for tc in ragas_tests:
    q   = tc["question"]
    # Giả định các hàm full_rag_answer và hybrid_search đã được định nghĩa ở trên
    ans = full_rag_answer(q, verbose=False)
    ctx_str  = hybrid_search(q, top_k=3, use_hyde=False, rerank=True)
    ctx_docs = [b.strip() for b in ctx_str.split("---") if len(b.strip()) > 20]

    questions.append(q)
    answers.append(ans)
    contexts_list.append(ctx_docs)
    ground_truths.append(tc["ground_truth"])

ds = Dataset.from_dict({
    "question": questions,
    "answer": answers,
    "contexts": contexts_list,
    "ground_truth": ground_truths
})

# --- 4. THỰC THI ĐÁNH GIÁ ---
if os.environ.get("GOOGLE_API_KEY"):
    print("🚀 Đang tiến hành đánh giá Ragas bằng Gemini...")
    # Truyền trực tiếp các object llm và embeddings vào hàm evaluate
    result = evaluate(
        ds,
        metrics=[faithfulness, answer_relevancy, context_precision],
        llm=gemini_llm,
        embeddings=gemini_embeddings
    )
    print("Ragas Result:", result)
else:
    print("⚠️ Setup thất bại: GOOGLE_API_KEY chưa được gán.")

ModuleNotFoundError: No module named 'langchain_core.pydantic_v1'